# SeqSFG playground: the current matched-incidence task

**Hear and inspect exactly what the current generator produces.** This version uses the
banded, matched-incidence setup: 45 ms tones, 5 ms ramps, seven components, nine elements,
26 tones per channel, and four-second intervals. All actual values below are read from
`pilot_config.json`, not a separate notebook configuration.

[Open in Colab](https://colab.research.google.com/github/MeysamAmirsardari/SeqSFG_task/blob/main/notebooks/SeqSFG_playground.ipynb)

Choose **Runtime → Run all** on a CPU runtime. Then use the audio players and interactive controls.
No audio autoplays. Use headphones at a comfortable, fixed volume. The isolated components are
explanatory views, not an exposure/training stage of the experiment. Full mixtures always retain
the configured background level. Downloads contain WAV files, plots, and reproduction metadata.

The notebook pins the current source revision for reproducibility. A local run uses your local
checkout and prints its commit and source hash. It does not silently switch an existing checkout.

In [ ]:
#@title Setup: load the current task and its configuration
import sys, os, subprocess, importlib.util, json, hashlib, io, wave, zipfile
from pathlib import Path
SOURCE_REF='edefc9fa96954b434d7e795b86c18e71c59c5c78'
REPO_URL='https://github.com/MeysamAmirsardari/SeqSFG_task.git'
IN_COLAB=bool(importlib.util.find_spec('google') and importlib.util.find_spec('google.colab'))
if IN_COLAB:
    ROOT=Path('/content')/('seqsfg-playground-'+SOURCE_REF[:12])
    if not ROOT.exists():
        subprocess.run(['git','clone','--no-checkout',REPO_URL,str(ROOT)],check=True)
        subprocess.run(['git','-C',str(ROOT),'checkout','--detach',SOURCE_REF],check=True)
    if subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()!=SOURCE_REF:
        raise RuntimeError('The existing clone has another revision. Start a fresh Colab runtime.')
else:
    ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'seqsfg/stimulus.py').exists()),None)
    if ROOT is None:raise RuntimeError('Run from the repository, or open in Colab.')
missing=[m for m in ['numpy','scipy','matplotlib','ipywidgets'] if importlib.util.find_spec(m) is None]
if missing:subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
sys.path.insert(0,str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Audio, Markdown, HTML, FileLink, clear_output
if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()
from seqsfg.config import Config, validate
from seqsfg.stimulus import make_trial, render_interval, render_trial, FIGURE, BACKGROUND, check_invariants
from seqsfg.session import source_hash
from seqsfg import measure, verify
cfg=Config.from_dict(json.loads((ROOT/'pilot_config.json').read_text()))
D=validate(cfg);SR=cfg.sample_rate
if not cfg.matched_incidence or cfg.figure_repeats!=1:
    raise ValueError('This playground visualizes matched_incidence=True and figure_repeats=1.')
COMMIT=subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
OUT=Path('/content/seqsfg_playground_outputs') if IN_COLAB else ROOT/'playground_outputs'
OUT.mkdir(exist_ok=True)
manifest={'commit':COMMIT,'source_hash':source_hash(),'config_hash':cfg.hash(),'config':cfg.to_dict(),'artifacts':{}}
print('Source:',COMMIT,'  source hash:',manifest['source_hash'])
print('Configuration:',cfg.hash())
print(f'{D.n_channels} channels; {cfg.n_components} components; {cfg.n_elements} elements; '
      f'{cfg.tones_per_channel} tones/channel; {cfg.interval_dur_ms/1000:g} seconds/interval')
print(f'Tones {cfg.tone_dur_ms:g} ms; ramps {cfg.ramp_ms:g} ms; mean simultaneous tones {D.mean_simultaneous:.2f}')
print('Main conditions:',cfg.main_variants,'  steps (ms):',cfg.steps_ms)
print('Figure band width (channels):',cfg.figure_band_channels,'  anchored fraction:',cfg.anchored_fraction)

## 1. What differs between the two intervals?

| Scheduled component | Target interval | Foil interval |
|---|---|---|
| The recurring frequency set S | Follows the selected within-element order | Independently scattered within each element window |
| Changing frequency sets F₁, F₂, … | Independently scattered | Follow the selected within-element order |
| Remaining tones | Fill each channel's fixed budget | Fill the same per-channel budget |

**S recurs in both intervals.** The question is which interval organizes the recurring set into
the selected temporal pattern. Both main-task intervals contain structured elements. Scattered
tones are not absent, and scattering does not guarantee that perceptual grouping is abolished.

`rising` uses ascending frequency order in every element. `redrawn` reshuffles the component-delay
order for each element in both intervals; the target's frequency set still recurs. At step zero,
order has no physical meaning, although separate random draws need not produce identical WAVs.

Each element is confined to a frequency band. The banded foil generator avoids S and tries to
separate consecutive foil elements in register and membership; it has fallback constraints, so
**do not assume consecutive foil sets are disjoint**. The actual overlaps are displayed below.

Approximately half of trials use one anchored S; the remainder use fresh sets. This is not exact
balance within every condition, nor a pure learned-order manipulation: frequency familiarity changes too.

In [ ]:
#@title Helpers: exact decomposition, fixed-gain audio, and plots
COLORS={'aligned':'#087f8c','scattered':'#dc7731','background':'#afb7c1'}
def subset(iv,indices):
    out=iv.copy()
    for key in ['onset','channel','phase','kind','element','component']:
        setattr(out,key,getattr(iv,key)[indices])
    return out

def decomposition(iv):
    # In this pinned build_matched implementation _assemble stores aligned tones first,
    # then scattered counterparts, then budget-filling background. Scattered tones are
    # labelled BACKGROUND in the package; kind alone cannot distinguish them.
    aligned=np.flatnonzero(iv.kind==FIGURE)
    bg=np.flatnonzero(iv.kind==BACKGROUND)
    n_scattered=cfg.n_elements*len(iv.figure_set)*(1 if len(aligned) else 2)
    assert len(bg)>=n_scattered
    return {'aligned':aligned,'scattered':bg[:n_scattered],'background':bg[n_scattered:]}

def save_audio(x,name,metadata=None):
    x=np.asarray(x,dtype=float)
    if not np.isfinite(x).all() or np.max(np.abs(x))>=1:raise ValueError('Nonfinite or clipping audio.')
    pcm=np.rint(x*32767).astype('<i2')
    path=OUT/(name+'.wav')
    with wave.open(str(path),'wb') as w:
        w.setnchannels(1);w.setsampwidth(2);w.setframerate(SR);w.writeframes(pcm.tobytes())
    manifest['artifacts'][path.name]={'sha256':hashlib.sha256(path.read_bytes()).hexdigest(),**(metadata or {})}
    # Playback uses the exact exported PCM samples; neither playback nor file is normalized.
    display(Audio(filename=str(path)))
    display(FileLink(str(path)))
    return path

def plot_raster(tr,name,zoom=False):
    fig,axes=plt.subplots(1,2,figsize=(13,4.3),sharey=True)
    for ax,iv,label in zip(axes,[tr.recurring,tr.other],['Target interval','Foil interval']):
        for role,idx in decomposition(iv).items():
            ax.hlines(iv.channel[idx],iv.onset[idx]*cfg.grid_ms,
                      iv.onset[idx]*cfg.grid_ms+cfg.tone_dur_ms,
                      colors=COLORS[role],linewidth=2 if role!='background' else .65,
                      alpha=.95 if role!='background' else .35,label=role)
        if zoom:
            start=iv.element_onsets[0]*cfg.grid_ms
            ax.set_xlim(start-15,start+cfg.iei_min_ms)
        else:ax.set_xlim(0,cfg.interval_dur_ms)
        ax.set(xlabel='Time (ms)',title=label)
        ticks=np.arange(0,D.n_channels,4)
        ax.set_yticks(ticks,[f'{D.channel_freqs_hz[k]:.0f}' for k in ticks])
    axes[0].set_ylabel('Channel frequency (Hz; ERB-spaced rows)')
    axes[1].legend(loc='upper right',fontsize=8)
    fig.suptitle(f'{tr.variant} | step {tr.step_ms:g} ms | seed {tr.seed} | anchored: {tr.anchored}')
    fig.tight_layout();fig.savefig(OUT/(name+'.png'),dpi=150);display(fig);plt.close(fig)

def show_pair(step=0,variant='rising',seed=102,target_position=1,details=False):
    tr=make_trial(cfg,int(seed),float(step),variant,d=D)
    base=f'{variant}_step{step:g}_seed{seed}'
    meta=dict(seed=int(seed),step_ms=float(step),variant=variant,anchored=bool(tr.anchored),rebuilds=tr.n_rebuilds)
    plot_raster(tr,base+'_full')
    plot_raster(tr,base+'_first_element',zoom=True)
    print('S frequencies (Hz):',np.round(D.channel_freqs_hz[tr.recurring.figure_set]).astype(int).tolist())
    fs=tr.other.element_sets
    if fs:
        print('Foil consecutive shared-channel counts:',[len(np.intersect1d(a,b)) for a,b in zip(fs[:-1],fs[1:])])
        print('Foil overlap with S:',[len(np.intersect1d(tr.recurring.figure_set,s)) for s in fs])
    display(Markdown(f'**Actual two-interval trial — target is interval {target_position}.**'))
    save_audio(render_trial(cfg,tr,target_position,d=D),base+'_whole',dict(meta,target_position=target_position))
    for label,iv in [('target',tr.recurring),('foil',tr.other)]:
        display(Markdown('**'+label.capitalize()+' — full mixture**'))
        save_audio(render_interval(cfg,iv,D),base+'_'+label,dict(meta,part='full',interval=label))
        if details:
            for part,idx in decomposition(iv).items():
                display(Markdown('**'+label.capitalize()+' — '+part+' tones only (explanatory isolation)**'))
                save_audio(render_interval(cfg,subset(iv,idx),D),base+'_'+label+'_'+part,dict(meta,part=part,interval=label))
    return tr

In [ ]:
#@title The synchronous task: full mixtures and their actual components
reference=show_pair(step=cfg.steps_ms[0],variant='rising',seed=102,details=True)

## 2. Explore the actual onset ladder and both main conditions

The whole-trial player includes the configured leading silence and 400 ms interval gap.
The colored plots include the structured group, its scattered counterpart, and all remaining
tones. The zoom shows actual tone durations, not just onset dots.

Changing the step also changes the element span, the scattering window, and the shared onset
jitter range. It is therefore not a manipulation of onset synchrony alone. The table is derived
from the current configuration. Positive adjacent overlap does not establish perceptual binding.

In [ ]:
#@title The timing geometry, calculated from the loaded setup
print('step   element span   matched footprint bound   adjacent overlap   max structured simultaneous')
for step in cfg.steps_ms:
    span=(cfg.n_components-1)*step+cfg.tone_dur_ms
    overlap=max(0,1-step/cfg.tone_dur_ms)
    concurrent=cfg.n_components if step==0 else min(cfg.n_components,int(np.ceil(cfg.tone_dur_ms/step)))
    print(f'{step:4g} ms   {span:6g} ms         {2*span:6g} ms                 {overlap:5.2f}              {concurrent}')
fig,axes=plt.subplots(len(cfg.steps_ms),2,figsize=(10,10),sharex=True,sharey=True)
for row,step in enumerate(cfg.steps_ms):
    for col,variant in enumerate(cfg.main_variants):
        tr=make_trial(cfg,104,float(step),variant,d=D)
        iv=tr.recurring; idx=np.flatnonzero((iv.kind==FIGURE)&(iv.element==0))
        times=iv.onset[idx]*cfg.grid_ms;times=times-times.min()
        axes[row,col].hlines(iv.component[idx],times,times+cfg.tone_dur_ms,color=COLORS['aligned'],linewidth=4)
        axes[row,col].set_title(f'{variant}: {step:g} ms',fontsize=10)
        axes[row,col].set_yticks(range(cfg.n_components));axes[row,col].grid(axis='x',alpha=.2)
axes[-1,0].set_xlabel('Time from first structured onset (ms)');axes[-1,1].set_xlabel('Time from first structured onset (ms)')
axes[2,0].set_ylabel('Component rank in frequency')
fig.suptitle('First structured element only; shared jitter removed for comparison')
fig.tight_layout();fig.savefig(OUT/'onset_ladder.png',dpi=150);display(fig);plt.close(fig)

In [ ]:
#@title Interactive audio and full rasters — select, then Build
step_control=widgets.Dropdown(options=list(cfg.steps_ms),value=cfg.steps_ms[1],description='Step (ms)')
variant_control=widgets.Dropdown(options=list(cfg.main_variants),description='Condition')
seed_control=widgets.IntText(value=102,description='Seed')
details_control=widgets.Checkbox(value=False,description='Isolated components')
build=widgets.Button(description='Build plots and audio',button_style='primary');explorer_out=widgets.Output()
def on_build(_):
    build.disabled=True
    with explorer_out:
        clear_output(wait=True)
        try:show_pair(step_control.value,variant_control.value,seed_control.value,details=details_control.value)
        except Exception as e:print(type(e).__name__+': '+str(e))
    build.disabled=False
build.on_click(on_build)
display(widgets.VBox([widgets.HBox([step_control,variant_control,seed_control]),details_control,build,explorer_out]))

## 3. Anchored versus fresh figures

The anchored set is selected by the configured seed; fresh sets vary across trials. Both main
conditions still use the same within-trial target/foil construction. This demonstration searches
for actual trials of each type instead of silently forcing an anchor. Repetition across trials
can support familiarity and attention; it is not by itself evidence for implicit order learning.

In [ ]:
#@title Show the actual frequency sets across anchored and fresh trials
examples={True:[],False:[]}
for seed in range(100,180):
    tr=make_trial(cfg,seed,0,'rising',d=D)
    if len(examples[tr.anchored])<3:examples[tr.anchored].append(tr)
    if all(len(v)==3 for v in examples.values()):break
fig,axes=plt.subplots(1,2,figsize=(10,3.2),sharey=True)
for ax,flag in zip(axes,[True,False]):
    for j,tr in enumerate(examples[flag]):
        ax.scatter(np.full(len(tr.recurring.figure_set),j),D.channel_freqs_hz[tr.recurring.figure_set],s=35)
    ax.set(title='Anchored S' if flag else 'Fresh S',xticks=range(len(examples[flag])),
           xticklabels=[f'seed {t.seed}' for t in examples[flag]],yscale='log')
axes[0].set_ylabel('Frequency (Hz)');fig.tight_layout();fig.savefig(OUT/'anchored_vs_fresh.png',dpi=150);display(fig);plt.close(fig)

## 4. What is physically matched, and what is not certified?

Equal per-channel tone budgets match scheduled occupancy. Finite waveforms can still show small
energy differences from phase interference. The graphs below are measurements of one generated
pair, not a population-level cue audit. Element-incidence matching does not make full per-channel
onset histories identical. Spectral register, temporal structure, and learned templates still
need appropriate observers.

In [ ]:
#@title Measure the pair: channel counts, broadband envelope, and spectrum
tr=make_trial(cfg,4242,7,'rising',d=D)
xa,xb=[render_interval(cfg,iv,D) for iv in [tr.recurring,tr.other]]
fig,axes=plt.subplots(1,3,figsize=(14,3.7))
for iv,x,label in [(tr.recurring,xa,'Target'),(tr.other,xb,'Foil')]:
    axes[0].plot(D.channel_freqs_hz,np.bincount(iv.channel,minlength=D.n_channels),'o-',label=label,alpha=.75)
    e=measure.frame_rms(x,SR,4)
    axes[1].plot(np.arange(len(e))*.004,e,label=label,alpha=.7)
    f=np.fft.rfftfreq(len(x),1/SR);power=np.abs(np.fft.rfft(x))**2
    axes[2].plot(f,10*np.log10(np.maximum(power,1e-12)),label=label,alpha=.6,linewidth=.5)
axes[0].set(xscale='log',xlabel='Frequency (Hz)',ylabel='Scheduled tone count',title='Per-channel budgets')
axes[1].set(xlabel='Time (s)',ylabel='4 ms RMS',title='Full-mixture envelopes')
axes[2].set(xscale='log',xlim=(150,12000),xlabel='Frequency (Hz)',ylabel='Power (arbitrary dB)',title='Full-interval spectra')
for ax in axes:ax.legend()
fig.tight_layout();fig.savefig(OUT/'physical_measurements.png',dpi=150);display(fig);plt.close(fig)
print('Schedule checks:',check_invariants(cfg,tr,D))

In [ ]:
#@title Validate decomposition and current stimulus invariants
for variant in list(cfg.main_variants)+['ungrouped','onechannel']:
    for step in ([cfg.steps_ms[0],cfg.steps_ms[-1]] if variant in cfg.main_variants else [0]):
        tr=make_trial(cfg,112,float(step),variant,d=D)
        checks=check_invariants(cfg,tr,D)
        for k in ['same_n_tones','same_channel_counts','budget_exact','no_same_channel_overlap']:
            assert checks[k],(variant,step,k)
        for iv in [tr.recurring,tr.other]:
            parts=decomposition(iv)
            idx=np.concatenate(list(parts.values()))
            assert np.array_equal(np.sort(idx),np.arange(iv.n_tones))
            full=render_interval(cfg,iv,D)
            summed=sum(render_interval(cfg,subset(iv,j),D).astype(float) for j in parts.values())
            assert np.allclose(full,summed,atol=1e-7)
            assert np.isfinite(full).all() and np.max(np.abs(full))<1
print('PASS: counts, budgets, same-channel nonoverlap, exact decomposition, and finite unclipped audio.')
print('These tests do not certify human audibility or absence of perceptual shortcuts.')

## 5. Try a short two-interval block

Choose the interval containing the recurring group. The browser plays both intervals, then
unlocks the response buttons. Feedback is optional; replay is logged. Download responses before
clearing the output. This mini-block uses the actual generator and gains, but browser timing,
and replay differ from the laboratory runner. The inter-interval gap uses the configured value. It is a listening
demonstration, not a replacement for a calibrated recording session.

In [ ]:
#@title Browser trial interface — run once
"""Colab-compatible browser trial panel; no callbacks, network requests, or autoplay.

Usage:
    display(browser_trial_panel(rows, sample_rate=cfg.sample_rate, provenance={...}))

Each row requires:
    trial_id: unique string/int
    audio_1, audio_2: nonempty one-dimensional mono float arrays in [-1, 1]
    correct_interval: integer 1 or 2
    phase: 'practice' (immediate feedback) or 'test' (no trial feedback)
Optional:
    instruction: participant-facing task instruction (no answer/condition leakage)
    condition: a condition identifier for final descriptive summaries
All remaining row fields are exported as metadata. Arrays are encoded as PCM16 WAV
without normalization; use the same physical rendering gain across all conditions.
Provenance should contain the repo commit, configuration/hash, seed and task version.
The returned HTML stores responses only in its browser output until CSV download.
Rerunning a panel creates a new session; export the old panel before clearing it.
"""
import base64
import io
import json
import uuid
import wave
from datetime import datetime, timezone

import numpy as np
from IPython.display import HTML


def browser_trial_panel(trial_rows, sample_rate, provenance):
    """Return self-contained HTML for a two-interval browser demonstration.

    Answers unlock only after both intervals have finished. Replaying is allowed
    before answering and logged. Test feedback is withheld until the end-of-block
    descriptive summary. RT is a browser estimate, not calibrated reaction time.
    """
    sample_rate = int(sample_rate)
    if sample_rate < 8000 or sample_rate > 192000:
        raise ValueError('sample_rate must be between 8,000 and 192,000 Hz')

    def json_default(value):
        if isinstance(value, np.ndarray):
            return value.tolist()
        if isinstance(value, np.generic):
            return value.item()
        raise TypeError(f'Metadata must be JSON serializable: {type(value).__name__}')

    def encode_audio(values):
        arr = np.asarray(values, dtype=float)
        if arr.ndim != 1 or arr.size == 0 or not np.isfinite(arr).all():
            raise ValueError('Audio must be a finite, nonempty mono array')
        if np.max(np.abs(arr)) > 1.0:
            raise ValueError('Audio exceeds [-1, 1]; set one common rendering gain before building the panel')
        stream = io.BytesIO()
        with wave.open(stream, 'wb') as wav:
            wav.setnchannels(1)
            wav.setsampwidth(2)
            wav.setframerate(sample_rate)
            wav.writeframes(np.rint(arr * 32767).astype('<i2').tobytes())
        return base64.b64encode(stream.getvalue()).decode('ascii')

    packed = []
    identifiers = set()
    for raw in trial_rows:
        row = dict(raw)
        trial_id = str(row['trial_id'])
        if trial_id in identifiers:
            raise ValueError(f'Duplicate trial_id: {trial_id}')
        identifiers.add(trial_id)
        if row['correct_interval'] not in (1, 2):
            raise ValueError('correct_interval must be 1 or 2')
        if row.get('phase') not in ('practice', 'test'):
            raise ValueError("phase must be 'practice' or 'test'")
        audio1 = encode_audio(row.pop('audio_1'))
        audio2 = encode_audio(row.pop('audio_2'))
        packed.append({'meta': row, 'audio': [audio1, audio2]})
    if not packed:
        raise ValueError('At least one trial is required')

    payload = {'trials': packed, 'sample_rate': sample_rate,
               'provenance': provenance,
               'created_utc': datetime.now(timezone.utc).isoformat()}
    data = json.dumps(payload, default=json_default, allow_nan=False)
    # Prevent metadata from ending the script element or injecting markup.
    data = data.replace('<', '\\u003c').replace('>', '\\u003e').replace('&', '\\u0026')
    data = data.replace('\u2028', '\\u2028').replace('\u2029', '\\u2029')
    panel_id = 'seqsfg_' + uuid.uuid4().hex
    template = r'''<div id="__PANEL_ID__" class="seqsfg-panel"></div>
<style>
#__PANEL_ID__ {max-width:820px;padding:22px;border:1px solid #bfd0df;border-radius:14px;background:#f8fbfe;color:#172d40;font:15px/1.5 system-ui,sans-serif;}
#__PANEL_ID__ h3 {margin:0 0 8px;font-size:22px;}
#__PANEL_ID__ p {margin:8px 0;}
#__PANEL_ID__ button {font:inherit;border:1px solid #426a8b;border-radius:7px;padding:10px 16px;margin:5px 8px 5px 0;background:white;color:#153d5b;cursor:pointer;}
#__PANEL_ID__ button:disabled {opacity:.45;cursor:default;}
#__PANEL_ID__ .primary {background:#155f91;color:white;}
#__PANEL_ID__ .status {min-height:48px;padding:12px;background:#e6f0f8;border-radius:8px;}
#__PANEL_ID__ .small {font-size:12px;color:#455d70;}
#__PANEL_ID__ table {border-collapse:collapse;margin-top:14px;width:100%;font-size:13px;}
#__PANEL_ID__ th, #__PANEL_ID__ td {text-align:left;border-bottom:1px solid #ccd7df;padding:7px;}
#__PANEL_ID__ progress {width:100%;height:12px;}
</style>
<script>
(() => {
  'use strict';
  const data = __PAYLOAD__;
  const root = document.getElementById('__PANEL_ID__');
  if (!root || root.dataset.initialized) return;
  root.dataset.initialized = 'true';
  const el = (tag, text, cls) => {
    const node = document.createElement(tag);
    if (text !== undefined) node.textContent = String(text);
    if (cls) node.className = cls;
    return node;
  };
  const title = el('h3', 'Listen and choose'); root.appendChild(title);
  root.appendChild(el('p', 'Which interval contains the recurring figure? Listen to both intervals, then choose 1 or 2. Start at a comfortable volume.'));
  const instruction = el('p'); root.appendChild(instruction);
  const progressText = el('p'); root.appendChild(progressText);
  const progress = el('progress'); progress.max = data.trials.length; progress.value = 0; root.appendChild(progress);
  const gainWrap = el('p');
  const gainLabel = el('label', 'Playback level: ');
  const gainSlider = el('input'); gainSlider.type = 'range'; gainSlider.min = '0.01'; gainSlider.max = '1'; gainSlider.step = '0.01'; gainSlider.value = '0.20'; gainSlider.setAttribute('aria-label', 'Playback level');
  const gainText = el('span', ' 20%'); gainLabel.append(gainSlider, gainText); gainWrap.appendChild(gainLabel); root.appendChild(gainWrap);
  const controls = el('div');
  const playButton = el('button', 'Play intervals 1 → 2', 'primary');
  const answer1 = el('button', 'Choose interval 1');
  const answer2 = el('button', 'Choose interval 2');
  const nextButton = el('button', 'Next trial');
  controls.append(playButton, answer1, answer2, nextButton); root.appendChild(controls);
  const status = el('p', 'Click Play to begin.', 'status'); status.setAttribute('role', 'status'); status.setAttribute('aria-live', 'polite'); root.appendChild(status);
  const score = el('p'); root.appendChild(score);
  const exportButton = el('button', 'Download responses CSV'); exportButton.disabled = true; root.appendChild(exportButton);
  root.appendChild(el('p', 'Demonstration only. Timing and sound level are not calibrated. RT is estimated by this browser from the last playback ending. Responses stay in this output until you download them; rerunning or clearing the output does not save them.', 'small'));
  const summary = el('div'); root.appendChild(summary);

  let index = 0, context = null, outputGain = null, decoded = null;
  let playing = false, answered = false, plays = 0, firstEnd = null, lastEnd = null;
  let firstStart = null, presentations = [], activeSources = [], intervalTimer = null;
  let interruptedPlays = 0, hiddenDuringTrial = false, playbackToken = 0;
  const responses = [];
  const sessionId = (globalThis.crypto && crypto.randomUUID) ? crypto.randomUUID() : ('browser-' + Date.now() + '-' + Math.random().toString(16).slice(2));
  const sessionStart = new Date().toISOString();

  function showTrial() {
    const row = data.trials[index].meta;
    answered = false; playing = false; plays = 0; decoded = null;
    firstEnd = null; lastEnd = null; firstStart = null; presentations = [];
    interruptedPlays = 0; hiddenDuringTrial = false;
    instruction.textContent = row.instruction || '';
    progressText.textContent = 'Trial ' + (index + 1) + ' of ' + data.trials.length + (row.phase === 'practice' ? ' · practice with feedback' : ' · test, feedback at end');
    progress.value = responses.length;
    playButton.textContent = 'Play intervals 1 → 2'; playButton.disabled = false;
    answer1.disabled = true; answer2.disabled = true; nextButton.disabled = true;
    gainSlider.disabled = false; status.textContent = 'Click Play when ready.';
    score.textContent = '';
  }

  function decodeWav(b64) {
    const raw = atob(b64), bytes = new Uint8Array(raw.length);
    for (let k = 0; k < raw.length; k++) bytes[k] = raw.charCodeAt(k);
    return context.decodeAudioData(bytes.buffer);
  }

  async function playPair() {
    if (playing || answered || index >= data.trials.length) return;
    playing = true;
    const ownToken = ++playbackToken;
    playButton.disabled = true; answer1.disabled = true; answer2.disabled = true; gainSlider.disabled = true;
    status.textContent = 'Preparing audio…';
    let thisPresentation = null;
    try {
      if (!context) {
        const AC = window.AudioContext || window.webkitAudioContext;
        if (!AC) throw new Error('Web Audio is unavailable in this browser.');
        context = new AC(); outputGain = context.createGain(); outputGain.connect(context.destination);
      }
      await context.resume();
      if (context.state !== 'running') throw new Error('Audio could not start. Keep this output visible and click Play again.');
      if (!decoded) decoded = await Promise.all(data.trials[index].audio.map(decodeWav));
      if (ownToken !== playbackToken || !playing) return;
      outputGain.gain.value = Number(gainSlider.value);
      const start = context.currentTime + 0.08;
      const second = start + decoded[0].duration + Number(data.provenance.config.isi_ms) / 1000;
      thisPresentation = {presentation_number: presentations.length + 1, gain: Number(gainSlider.value), started_utc: new Date().toISOString(), browser_start_ms: performance.now(), context_start_s: start, interval2_start_s: second, context_end_s: second + decoded[1].duration, completed: false};
      presentations.push(thisPresentation);
      if (firstStart === null) firstStart = thisPresentation.browser_start_ms;
      status.textContent = 'Playing interval 1…';
      intervalTimer = setTimeout(() => { if (playing) status.textContent = 'Playing interval 2…'; }, Math.max(0, (second - context.currentTime) * 1000));
      const source1 = context.createBufferSource(), source2 = context.createBufferSource();
      source1.buffer = decoded[0]; source2.buffer = decoded[1];
      source1.connect(outputGain); source2.connect(outputGain); activeSources = [source1, source2];
      source2.onended = () => {
        if (!playing) return;
        clearTimeout(intervalTimer); activeSources = []; playing = false; plays += 1;
        lastEnd = performance.now(); if (firstEnd === null) firstEnd = lastEnd;
        thisPresentation.completed = true; thisPresentation.browser_ended_ms = lastEnd;
        playButton.textContent = 'Replay both intervals'; playButton.disabled = false;
        answer1.disabled = false; answer2.disabled = false; gainSlider.disabled = false;
        status.textContent = 'Choose interval 1 or 2. You may replay both before answering.';
      };
      source1.start(start); source2.start(second);
    } catch (error) {
      playing = false; clearTimeout(intervalTimer);
      for (const source of activeSources) { source.onended = null; try {source.stop();} catch (_) {} }
      activeSources = [];
      if (thisPresentation) thisPresentation.error = String(error.message || error);
      status.textContent = 'Audio could not play: ' + String(error.message || error) + ' Click Play to retry.';
      playButton.disabled = false; gainSlider.disabled = false;
      answer1.disabled = lastEnd === null; answer2.disabled = lastEnd === null;
    }
  }

  function answer(choice) {
    if (playing || answered || lastEnd === null) return;
    answered = true;
    const now = performance.now(), row = data.trials[index].meta;
    const correct = Number(choice === Number(row.correct_interval));
    const record = {};
    // Prefixes keep user metadata and measured response fields from overwriting each other.
    Object.keys(row).forEach(key => { record['meta_' + key] = row[key]; });
    Object.assign(record, {session_id: sessionId, session_started_utc: sessionStart,
      trial_number: index + 1, choice: choice, correct: correct,
      response_utc: new Date().toISOString(), rt_browser_ms_since_last_pair_end: Math.round(now-lastEnd),
      browser_ms_since_first_pair_end: Math.round(now-firstEnd), browser_trial_elapsed_ms: Math.round(now-firstStart),
      completed_pair_plays: plays, replay_count: Math.max(0, plays-1),
      interrupted_playbacks: interruptedPlays, document_hidden_during_trial: hiddenDuringTrial,
      playback_gain_at_response: Number(gainSlider.value), presentation_log: presentations,
      wav_sample_rate: data.sample_rate, audio_context_sample_rate: context.sampleRate,
      browser_user_agent: navigator.userAgent, panel_created_utc: data.created_utc,
      provenance_json: data.provenance,
      timing_note: 'Approximate performance.now timestamp after Web Audio onended; uncalibrated output latency; RT restarts after replay.'});
    responses.push(record); progress.value = responses.length;
    answer1.disabled = true; answer2.disabled = true; playButton.disabled = true; gainSlider.disabled = true;
    nextButton.disabled = false; exportButton.disabled = false;
    if (row.phase === 'practice') {
      status.textContent = correct ? 'Correct — interval ' + row.correct_interval + '.' : 'The recurring figure was in interval ' + row.correct_interval + '.';
      const practice = responses.filter(r => r.meta_phase === 'practice');
      score.textContent = 'Practice: ' + practice.reduce((a, r) => a + r.correct, 0) + ' / ' + practice.length + ' correct.';
    } else {
      status.textContent = 'Response saved. Continue when ready.'; score.textContent = '';
    }
    nextButton.textContent = index + 1 === data.trials.length ? 'Finish and show summary' : 'Next trial';
  }

  function finish() {
    controls.style.display = 'none'; gainWrap.style.display = 'none'; instruction.textContent = '';
    progressText.textContent = 'Completed ' + responses.length + ' trials.';
    status.textContent = 'Block complete. Download your responses before clearing this output.';
    score.textContent = '';
    summary.appendChild(el('h3', 'Descriptive results'));
    const table = el('table'), head = el('tr');
    ['Phase', 'Condition', 'n', 'Proportion correct'].forEach(label => head.appendChild(el('th', label)));
    const thead = el('thead'); thead.appendChild(head); table.appendChild(thead);
    const tbody = el('tbody'), groups = new Map();
    responses.forEach(r => {
      const condition = r.meta_condition === undefined ? 'all trials' : (typeof r.meta_condition === 'object' ? JSON.stringify(r.meta_condition) : String(r.meta_condition));
      const key = JSON.stringify([r.meta_phase, condition]);
      if (!groups.has(key)) groups.set(key, {phase: r.meta_phase, condition, n: 0, correct: 0});
      const g = groups.get(key); g.n++; g.correct += r.correct;
    });
    groups.forEach(g => {const tr = el('tr'); [g.phase, g.condition, g.n, (g.correct/g.n).toFixed(3)].forEach(v => tr.appendChild(el('td', v))); tbody.appendChild(tr);});
    table.appendChild(tbody); summary.appendChild(table);
    summary.appendChild(el('p', 'Chance is 0.50. These few trials illustrate the task; they are not a formal experiment and cannot establish learning, binding, or a mechanism. Replays, device settings and uncontrolled listening conditions limit interpretation.', 'small'));
  }

  function csvValue(value) {
    let text = value === null || value === undefined ? '' : (typeof value === 'object' ? JSON.stringify(value) : String(value));
    // Quoting protects CSV structure; an apostrophe protects spreadsheet formula-like text.
    if (typeof value !== 'number' && /^[\s]*[=+@-]/.test(text)) text = "'" + text;
    return '"' + text.replace(/"/g, '""') + '"';
  }
  function download() {
    if (!responses.length) return;
    const keys = [...new Set(responses.flatMap(row => Object.keys(row)))];
    const lines = [keys.map(csvValue).join(','), ...responses.map(row => keys.map(key => csvValue(row[key])).join(','))];
    const blob = new Blob(['\uFEFF' + lines.join('\r\n')], {type: 'text/csv;charset=utf-8;'});
    const url = URL.createObjectURL(blob), link = el('a');
    link.href = url; link.download = 'SeqSFG_browser_demo_' + sessionId + '.csv';
    root.appendChild(link); link.click(); link.remove(); setTimeout(() => URL.revokeObjectURL(url), 15000);
  }
  document.addEventListener('visibilitychange', () => {
    if (!document.hidden || answered || index >= data.trials.length) return;
    hiddenDuringTrial = true;
    if (playing) {
      interruptedPlays++; playbackToken++; playing = false; clearTimeout(intervalTimer);
      for (const source of activeSources) {source.onended = null; try {source.stop();} catch (_) {} }
      activeSources = [];
      status.textContent = 'Playback stopped because this page was hidden. Replay both intervals before answering.';
      lastEnd = null; playButton.disabled = false; answer1.disabled = true; answer2.disabled = true; gainSlider.disabled = false;
    }
  });
  gainSlider.addEventListener('input', () => {gainText.textContent = ' ' + Math.round(Number(gainSlider.value)*100) + '%';});
  playButton.addEventListener('click', playPair);
  answer1.addEventListener('click', () => answer(1)); answer2.addEventListener('click', () => answer(2));
  nextButton.addEventListener('click', () => {if (!answered) return; index++; if (index === data.trials.length) finish(); else showTrial();});
  exportButton.addEventListener('click', download);
  showTrial();
})();
</script>'''
    return HTML(template.replace('__PANEL_ID__', panel_id).replace('__PAYLOAD__', data))

In [ ]:
#@title Interactive mini-block using the current generator
mini_step=widgets.Dropdown(options=list(cfg.steps_ms),description='Step (ms)')
mini_variant=widgets.Dropdown(options=list(cfg.main_variants),description='Condition')
mini_n=widgets.Dropdown(options=[4,8,12],value=4,description='Trials')
mini_feedback=widgets.Checkbox(value=True,description='Feedback')
mini_button=widgets.Button(description='Build listening block',button_style='primary');mini_out=widgets.Output()
def build_mini(step=0,variant='rising',n=4,feedback=True,seed=9301):
    rows=[];positions=[1,2]*(n//2);np.random.default_rng(seed).shuffle(positions)
    for i,target in enumerate(positions):
        tr=make_trial(cfg,seed+i,float(step),variant,d=D)
        a,b=[render_interval(cfg,iv,D) for iv in [tr.recurring,tr.other]]
        x,y=(a,b) if target==1 else (b,a)
        rows.append(dict(trial_id=seed+i,audio_1=x,audio_2=y,correct_interval=target,
                         phase='practice' if feedback else 'test',condition=variant,step_ms=float(step),
                         seed=seed+i,anchored=bool(tr.anchored),rebuilds=tr.n_rebuilds))
    return browser_trial_panel(rows,SR,dict(commit=COMMIT,source_hash=source_hash(),config=cfg.to_dict(),config_hash=cfg.hash()))
def on_mini(_):
    with mini_out:
        clear_output(wait=True)
        display(build_mini(mini_step.value,mini_variant.value,mini_n.value,mini_feedback.value))
mini_button.on_click(on_mini)
display(widgets.VBox([widgets.HBox([mini_step,mini_variant,mini_n]),mini_feedback,mini_button,mini_out]))

## 6. Listen to the controls without overinterpreting them

`ungrouped`: the target organizes S; the comparison scatters both scheduled sets. It is **not a
silence interval**. `onechannel`: only one target channel is used; no multi-component figure can
be inferred from success. These controls need their own measured cue profiles. Their small
trial counts cannot establish the absence of a usable cue.

In [ ]:
#@title Actual control audio: target and comparison, with complete backgrounds
for variant in ['ungrouped','onechannel']:
    tr=make_trial(cfg,909,0,variant,d=D)
    display(Markdown('**'+variant+' — target is interval 1**'))
    save_audio(render_trial(cfg,tr,1,D),variant+'_control_seed909',dict(seed=909,variant=variant,step_ms=0,target_position=1))

## 7. Optional fresh cue audit

The old saved pilot report may describe a different density or band configuration. It must not
be used as evidence for this setup unless its configuration hash matches. Run the live battery
to measure this setup. Small samples are exploratory; a nonsignificant result is not an
equivalence test and does not certify that all non-binding strategies are unavailable.

In [ ]:
#@title Check report provenance and optionally run the current battery
import re
report_path=ROOT/'verification/pilot_battery_report.txt'
if report_path.exists():
    text=report_path.read_text();match=re.search(r'config hash\s+([0-9a-f]+)',text)
    matches=bool(match and match.group(1)==cfg.hash())
    print('Saved report matches current configuration:',matches)
    if not matches:print('Ignore old report conclusions for this setup. Run a fresh audit below.')
audit_n=widgets.Dropdown(options=[12,40,80],value=40,description='Pairs/cell')
audit_button=widgets.Button(description='Run full cue battery',button_style='warning');audit_out=widgets.Output()
def on_audit(_):
    audit_button.disabled=True
    with audit_out:
        clear_output(wait=True);print('Running the exact current configuration; this may take a few minutes.')
        try:
            res=verify.run_battery(cfg,n_trials=audit_n.value,seed=20260910,verbose=False)
            report=verify.format_report(res);print(report)
            (OUT/'current_cue_report.txt').write_text(report)
            (OUT/'current_cue_audit.json').write_text(json.dumps(verify.to_json(res),indent=2))
        except Exception as e:print(type(e).__name__+': '+str(e))
    audit_button.disabled=False
audit_button.on_click(on_audit)
display(widgets.VBox([audit_n,audit_button,audit_out]))

## 8. Download the plots and audio

The bundle contains the WAV files and PNG figures generated by the cells/buttons you ran, plus
the complete configuration, source hash, commit, seeds, condition labels, and WAV checksums.
All WAVs share one rendering gain. Isolated parts are explicitly labelled. No headphone SPL
calibration is implied. The existing generator and experimental configuration are unchanged.

**What this playground establishes:** what is physically scheduled, what is played, and which
measurements/checks were actually run. Whether listeners bind the components or learn their
order requires behavioral data and appropriately controlled comparisons.

In [ ]:
#@title Download current plots, WAV files, and reproduction metadata
download=widgets.Button(description='Download plots and audio',button_style='success');download_out=widgets.Output()
def export_outputs():
    (OUT/'manifest.json').write_text(json.dumps(manifest,indent=2))
    path=OUT/'SeqSFG_current_playground.zip'
    with zipfile.ZipFile(path,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUT.iterdir()):
            if p.suffix in ['.wav','.png','.json','.txt']:z.write(p,p.name)
    return path
def on_download(_):
    with download_out:
        clear_output(wait=True);path=export_outputs()
        if IN_COLAB:
            from google.colab import files
            files.download(str(path))
        else:display(FileLink(str(path)))
download.on_click(on_download);display(widgets.VBox([download,download_out]))